In [1]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd
import numpy as np
import os
import gc

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, recall_score

In [3]:
BASE_PATH = "/content/drive/MyDrive/MajorProject"
# Use augmented dataset for training
TRAIN_PATH = f"{BASE_PATH}/CIC_IoT_2023/Full_Dataset/augmented_train_data.csv"
TEST_PATH = f"{BASE_PATH}/CIC_IoT_2023/Full_Dataset/test_frozen.csv"

LABEL_COL = "label"
RANDOM_SEED = 42

In [4]:
print("Loading Training data")
train_df = pd.read_csv(TRAIN_PATH, low_memory=False)

print("Loading Test data")
test_df = pd.read_csv(TEST_PATH, low_memory=False)

print("Datasets Loaded Succesfully")

Loading Training data
Loading Test data
Datasets Loaded Succesfully


In [5]:
train_df.sample(10)

,flow_duration,Header_Length,Protocol Type,Duration,Rate,Srate,Drate,fin_flag_number,syn_flag_number,rst_flag_number,...,Std,Tot size,IAT,Number,Magnitue,Radius,Covariance,Variance,Weight,label
775808,2.051733,3880094.00,6.00,64.00,1669.076842,1669.076842,0.0,0.0,0.0,0.0,...,0.000000,94.00,1.668625e+08,13.5,13.711309,0.000000,0.000000,0.00,244.60,DNS_Spoofing
5590,0.000000,54.24,6.00,64.00,4.761405,4.761405,0.0,0.0,0.0,0.0,...,0.160873,54.24,8.307636e+07,9.5,10.395997,0.229325,0.876627,0.03,141.55,DDoS-TCP_Flood
11760,0.021897,14201.00,17.00,64.00,13087.198866,13087.198866,0.0,0.0,0.0,0.0,...,0.000000,50.00,8.310283e+07,9.5,10.000000,0.000000,0.000000,0.00,141.55,DDoS-UDP_Flood
256772,0.000000,54.00,6.00,64.00,2.702253,2.702253,0.0,0.0,1.0,0.0,...,0.000000,54.00,8.298538e+07,9.5,10.392305,0.000000,0.000000,0.00,141.55,DoS-SYN_Flood
520259,7.717113,302343.09,6.22,79.93,22.450430,22.450430,0.0,0.0,1.0,0.0,...,239.952545,382.12,8.299935e+07,9.5,20.059954,339.406648,295849.674789,0.37,141.55,DoS-HTTP_Flood
73111,0.080807,5887.34,1.47,245.45,48.263300,48.263300,0.0,0.0,0.0,0.0,...,10.140413,72.13,8.300818e+07,9.5,12.120118,14.356425,609.532518,0.34,141.55,DoS-UDP_Flood
407470,0.000000,757.00,6.00,64.00,20.885316,20.885316,0.0,0.0,0.0,0.0,...,538.470740,944.00,8.333668e+07,9.5,44.788211,761.456760,305219.322301,0.95,141.55,DDoS-ACK_Fragmentation
890128,0.283809,120.98,6.00,64.00,5.919301,5.919301,0.0,0.0,0.0,1.0,...,0.560334,56.60,8.300283e+07,9.5,10.721232,0.788999,4.587628,0.07,141.55,DoS-HTTP_Flood
236035,5.550573,108.00,6.00,64.00,0.360323,0.360323,0.0,0.0,0.0,0.0,...,0.000000,54.00,8.295580e+07,9.5,10.392305,0.000000,0.000000,0.00,141.55,DoS-TCP_Flood
660539,0.081625,1122.50,8.20,138.80,151.565322,151.565322,0.0,0.0,0.0,0.0,...,125.464393,200.50,1.668618e+08,13.5,16.841463,177.772964,16819.659340,1.00,244.60,DNS_Spoofing


In [6]:
print(f"Shape of training data: {train_df.shape}")
print(f"Shape of test data: {test_df.shape}")

Shape of training data: (912851, 47)
Shape of test data: (259646, 47)


### Clean data

In [7]:
train_df = train_df.dropna(subset=[LABEL_COL])
test_df = test_df.dropna(subset=[LABEL_COL])

print("Train shape after cleaning", train_df.shape)
print("Test shape after cleaning", test_df.shape)

Train shape after cleaning (912851, 47)
Test shape after cleaning (259645, 47)


### Split features and target

In [8]:
x_train = train_df.drop(columns=[LABEL_COL])
y_train = train_df[LABEL_COL]

x_test = test_df.drop(columns=[LABEL_COL])
y_test = test_df[LABEL_COL]

del train_df, test_df
gc.collect()

0

In [9]:
x_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 912851 entries, 0 to 912850
Data columns (total 46 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   flow_duration    912851 non-null  float64
 1   Header_Length    912851 non-null  float64
 2   Protocol Type    912851 non-null  float64
 3   Duration         912851 non-null  float64
 4   Rate             912851 non-null  float64
 5   Srate            912851 non-null  float64
 6   Drate            912851 non-null  float64
 7   fin_flag_number  912851 non-null  float64
 8   syn_flag_number  912851 non-null  float64
 9   rst_flag_number  912851 non-null  float64
 10  psh_flag_number  912851 non-null  float64
 11  ack_flag_number  912851 non-null  float64
 12  ece_flag_number  912851 non-null  float64
 13  cwr_flag_number  912851 non-null  float64
 14  ack_count        912851 non-null  float64
 15  syn_count        912851 non-null  float64
 16  fin_count        912851 non-null  floa

In [10]:
print("NANs in x_train:", x_train.isna().sum().sum())
print("NaNs in X_test:", x_test.isna().sum().sum())
print("NaNs in y_train:", y_train.isna().sum())
print("NaNs in y_test:", y_test.isna().sum())

NANs in x_train: 0
NaNs in X_test: 0
NaNs in y_train: 0
NaNs in y_test: 0


### Encode labels

In [11]:
le = LabelEncoder()

y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

class_names = le.classes_

print("Number of classes", len(class_names))

Number of classes 34


### Define Random Forest

In [12]:
rf_model = RandomForestClassifier(
    n_estimators=150,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    n_jobs=-1,
    random_state=RANDOM_SEED,
    verbose=1
)

### Train the model

In [13]:
print("Training Random forest on", x_train.shape[0], "rows")

rf_model.fit(x_train, y_train_enc)

Training Random forest on 912851 rows


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done  46 tasks      | elapsed:  3.5min
[Parallel(n_jobs=-1)]: Done 150 out of 150 | elapsed: 11.6min finished


RandomForestClassifier(n_estimators=150, n_jobs=-1, random_state=42, verbose=1)

In [14]:
print("Generating predictions...")
y_pred_enc = rf_model.predict(x_test)

Generating predictions...


[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    5.5s
[Parallel(n_jobs=2)]: Done 150 out of 150 | elapsed:   14.5s finished


In [15]:
print("Generating predictions...")
y_pred_enc = rf_model.predict(x_test)

Generating predictions...


[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    6.6s
[Parallel(n_jobs=2)]: Done 150 out of 150 | elapsed:   15.0s finished


In [16]:
report = classification_report(
    y_test_enc,
    y_pred_enc,
    target_names=class_names,
    digits=4
)

print(report)

                         precision    recall  f1-score   support

       Backdoor_Malware     0.9576    0.5313    0.6835       638
          BenignTraffic     0.7612    0.7482    0.7546      6000
       BrowserHijacking     0.9748    0.4636    0.6283      1167
       CommandInjection     0.9444    0.6141    0.7442      1078
 DDoS-ACK_Fragmentation     0.9979    0.9968    0.9974      7799
        DDoS-HTTP_Flood     0.9962    0.9932    0.9947      5742
        DDoS-ICMP_Flood     1.0000    0.9990    0.9995      6000
DDoS-ICMP_Fragmentation     0.9984    0.9953    0.9969      7655
      DDoS-PSHACK_Flood     0.9998    0.9998    0.9998      6000
       DDoS-RSTFINFlood     1.0000    0.9993    0.9997      6000
         DDoS-SYN_Flood     0.9985    0.9968    0.9977      6000
         DDoS-SlowLoris     0.9840    0.9974    0.9906      4672
DDoS-SynonymousIP_Flood     0.9987    0.9982    0.9984      6000
         DDoS-TCP_Flood     0.9998    0.9983    0.9991      6000
         DDoS-UDP_Flood 

In [17]:
recalls = recall_score(
    y_test_enc,
    y_pred_enc,
    average=None
)

recall_df = pd.DataFrame({
    "Class": class_names,
    "Recall": recalls
}).sort_values("Recall")

recall_df

,Class,Recall
28,Recon-PingSweep,0.198661
31,Uploading_Attack,0.388000
30,SqlInjection,0.425287
2,BrowserHijacking,0.463582
33,XSS,0.522193
0,Backdoor_Malware,0.531348
17,DictionaryBruteForce,0.596309
3,CommandInjection,0.614100
29,Recon-PortScan,0.748003
1,BenignTraffic,0.748167


In [18]:
MODEL_B_OUTPUT_DIR = f"{BASE_PATH}/IDS_ModelB_Outputs"
os.makedirs(MODEL_B_OUTPUT_DIR, exist_ok=True)

recall_path = f"{MODEL_B_OUTPUT_DIR}/model_B_per_class_recall.csv"
recall_df.to_csv(recall_path, index=False)

print("✅ Per-class recall saved to:", recall_path)

✅ Per-class recall saved to: /content/drive/MyDrive/MajorProject/IDS_ModelB_Outputs/model_B_per_class_recall.csv


In [ ]:
import joblib

joblib.dump(rf_model, f"{MODEL_B_OUTPUT_DIR}/modelB_RF.joblib")
joblib.dump(le, f"{MODEL_B_OUTPUT_DIR}/modelB_labelEncoder.joblib")

print("Model B Saved succesfully")

Model B Saved succesfully
